In [6]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns


In [13]:
p = "Employer_Information.csv"
df = pd.read_csv(p, encoding="utf-16", sep="\t")
df.head(5)

,Line by line,Fiscal Year,Employer (Petitioner) Name,Tax ID,Industry (NAICS) Code,Petitioner City,Petitioner State,Petitioner Zip Code,New Employment Approval,New Employment Denial,Continuation Approval,Continuation Denial,Change with Same Employer Approval,Change with Same Employer Denial,New Concurrent Approval,New Concurrent Denial,Change of Employer Approval,Change of Employer Denial,Amended Approval,Amended Denial
0,1,2026,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,1,0,0,0,0,0,0,0
1,2,2026,NaN,NaN,NaN,DALLAS,TX,75275.0,0,0,0,0,0,0,0,0,0,0,0,0
2,3,2026,NaN,NaN,NaN,NEW ORLEANS,LA,70123.0,0,0,0,0,0,0,0,0,0,0,0,0
3,4,2026,NaN,NaN,NaN,RICHARDSON,TX,75081.0,0,0,0,0,0,0,0,0,0,0,0,0
4,5,2026,NaN,3227.0,"54 - Professional, Scientific, and Technical S...",CANTON,MI,48187.0,0,0,0,0,0,0,0,0,0,0,0,1


In [14]:
df.info

<bound method DataFrame.info of       Line by line  Fiscal Year    Employer (Petitioner) Name  Tax ID  \
0                1            2026                        NaN     NaN   
1                2            2026                        NaN     NaN   
2                3            2026                        NaN     NaN   
3                4            2026                        NaN     NaN   
4                5            2026                        NaN  3227.0   
...            ...             ...                        ...     ...   
65467       65,468            2026                  ZYTUS INC    96.0   
65468       65,469            2026                  ZYWIE INC  9321.0   
65469       65,470            2026                 ZYWIE INC.  9321.0   
65470       65,471            2026                 ZYYO, INC.  5558.0   
65471       65,472            2026         ZZOUNDS MUSIC, LLC  7798.0   

                                   Industry (NAICS) Code Petitioner City  \
0              

In [15]:
df.shape

(65472, 20)

In [20]:
df.columns

Index(['Line by line', 'Fiscal Year   ', 'Employer (Petitioner) Name',
       'Tax ID', 'Industry (NAICS) Code', 'Petitioner City',
       'Petitioner State', 'Petitioner Zip Code', 'New Employment Approval',
       'New Employment Denial', 'Continuation Approval', 'Continuation Denial',
       'Change with Same Employer Approval',
       'Change with Same Employer Denial', 'New Concurrent Approval',
       'New Concurrent Denial', 'Change of Employer Approval',
       'Change of Employer Denial', 'Amended Approval', 'Amended Denial'],
      dtype='str')

In [21]:
df.dtypes

Line by line                              str
Fiscal Year                             int64
Employer (Petitioner) Name                str
Tax ID                                float64
Industry (NAICS) Code                     str
Petitioner City                           str
Petitioner State                          str
Petitioner Zip Code                   float64
New Employment Approval                   str
New Employment Denial                   int64
Continuation Approval                     str
Continuation Denial                     int64
Change with Same Employer Approval        str
Change with Same Employer Denial        int64
New Concurrent Approval                 int64
New Concurrent Denial                   int64
Change of Employer Approval               str
Change of Employer Denial               int64
Amended Approval                          str
Amended Denial                          int64
dtype: object

In [23]:
df.dtypes.astype(str).to_dict()

{'Line by line': 'str',
 'Fiscal Year   ': 'int64',
 'Employer (Petitioner) Name': 'str',
 'Tax ID': 'float64',
 'Industry (NAICS) Code': 'str',
 'Petitioner City': 'str',
 'Petitioner State': 'str',
 'Petitioner Zip Code': 'float64',
 'New Employment Approval': 'str',
 'New Employment Denial': 'int64',
 'Continuation Approval': 'str',
 'Continuation Denial': 'int64',
 'Change with Same Employer Approval': 'str',
 'Change with Same Employer Denial': 'int64',
 'New Concurrent Approval': 'int64',
 'New Concurrent Denial': 'int64',
 'Change of Employer Approval': 'str',
 'Change of Employer Denial': 'int64',
 'Amended Approval': 'str',
 'Amended Denial': 'int64'}

## # normalize headers and inspect completeness, totals


In [24]:
df.columns=[c.strip().lower().replace(" ","_") for c in df.columns]
df["fiscal_year"]=pd.to_numeric(df["fiscal_year"],errors="coerce")
id_cols=["employer_(petitioner)_name","tax_id","industry_(naics)_code","petitioner_city","petitioner_state"]
{c: (df[c].notna().sum(), df[c].nunique(dropna=True)) for c in id_cols}, df["fiscal_year"].value_counts(dropna=False).to_dict(), df.tail(8).to_string(index=False)


({'employer_(petitioner)_name': (np.int64(65465), 57673),
  'tax_id': (np.int64(65330), 9965),
  'industry_(naics)_code': (np.int64(62057), 20),
  'petitioner_city': (np.int64(65471), 4994),
  'petitioner_state': (np.int64(65430), 57)},
 {2026: 65472},
 'line_by_line  fiscal_year employer_(petitioner)_name  tax_id                                 industry_(naics)_code petitioner_city petitioner_state  petitioner_zip_code new_employment_approval  new_employment_denial continuation_approval  continuation_denial change_with_same_employer_approval  change_with_same_employer_denial  new_concurrent_approval  new_concurrent_denial change_of_employer_approval  change_of_employer_denial amended_approval  amended_denial\n      65,465         2026              ZYSCOVICH LLC  4852.0 54 - Professional, Scientific, and Technical Services           MIAMI               FL              33132.0                       0                      0                     1                    0                      

## Convert count fields robustly


In [25]:
count_cols=[c for c in df.columns if c.endswith("_approval") or c.endswith("_denial")]
for c in count_cols:
    df[c]=pd.to_numeric(df[c].astype(str).str.replace(",","",regex=False), errors="coerce").fillna(0)
approval_cols=[c for c in count_cols if c.endswith("_approval")]
denial_cols=[c for c in count_cols if c.endswith("_denial")]
df["approvals"]=df[approval_cols].sum(axis=1)
df["denials"]=df[denial_cols].sum(axis=1)
df["decisions"]=df["approvals"]+df["denials"]
df["approval_rate"]=np.where(df["decisions"]>0,df["approvals"]/df["decisions"],np.nan)
df["employer_clean"]=(df["employer_(petitioner)_name"].astype(str).str.upper()
                      .str.replace(r"[^A-Z0-9 ]","",regex=True)
                      .str.replace(r"\b(INC|LLC|LTD|CORP|CORPORATION|COMPANY|CO)\b","",regex=True)
                      .str.replace(r"\s+"," ",regex=True).str.strip())
df["fiscal_year"]=pd.to_numeric(df["fiscal_year"],errors="coerce").astype("Int64")
summary=(df.groupby(["employer_clean","petitioner_state"],dropna=False)
         .agg(approvals=("approvals","sum"),denials=("denials","sum"),
              decisions=("decisions","sum"))
         .reset_index())
summary["approval_rate"]=np.where(summary.decisions>0,summary.approvals/summary.decisions,np.nan)
summary=summary.sort_values("approvals",ascending=False)
metrics={
"rows":len(df),
"years":sorted(df.fiscal_year.dropna().unique().tolist()),
"employers_nonnull":int(df["employer_(petitioner)_name"].notna().sum()),
"states":int(df.petitioner_state.nunique()),
"approvals":int(df.approvals.sum()),
"denials":int(df.denials.sum()),
"decisions":int(df.decisions.sum()),
"top10":summary.head(10)[["employer_clean","petitioner_state","approvals","denials","approval_rate"]].to_dict("records"),
"approval_columns":approval_cols,
"denial_columns":denial_cols,
"employer_name_missing":int(df["employer_(petitioner)_name"].isna().sum()),
}
metrics


{'rows': 65472,
 'years': [2026],
 'employers_nonnull': 65465,
 'states': 57,
 'approvals': 395176,
 'denials': 11990,
 'decisions': 407166,
 'top10': [{'employer_clean': 'AMAZON COM SERVICES',
   'petitioner_state': 'VA',
   'approvals': 12007,
   'denials': 134,
   'approval_rate': 0.9889630178733217},
  {'employer_clean': 'GOOGLE',
   'petitioner_state': 'CA',
   'approvals': 6163,
   'denials': 85,
   'approval_rate': 0.9863956466069143},
  {'employer_clean': 'TATA CONSULTANCY SERVICES LIMITED',
   'petitioner_state': 'MD',
   'approvals': 5539,
   'denials': 380,
   'approval_rate': 0.9357999662105085},
  {'employer_clean': 'APPLE',
   'petitioner_state': 'CA',
   'approvals': 5285,
   'denials': 55,
   'approval_rate': 0.9897003745318352},
  {'employer_clean': 'INFOSYS LIMITED',
   'petitioner_state': 'TX',
   'approvals': 4205,
   'denials': 129,
   'approval_rate': 0.9702353484079372},
  {'employer_clean': 'META PLATFORMS',
   'petitioner_state': 'CA',
   'approvals': 3996,
   

In [27]:
import os, re, zipfile

source = "Employer_Information.csv"
outdir = "h1b_powerbi_files"
os.makedirs(outdir, exist_ok=True)

df = pd.read_csv(source, encoding="utf-16", sep="\t")
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Convert USCIS petition decision fields to numeric
count_cols = [c for c in df.columns if c.endswith("_approval") or c.endswith("_denial")]
for c in count_cols:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False), errors="coerce").fillna(0)

approval_cols = [c for c in count_cols if c.endswith("_approval")]
denial_cols = [c for c in count_cols if c.endswith("_denial")]
df["approvals"] = df[approval_cols].sum(axis=1)
df["denials"] = df[denial_cols].sum(axis=1)
df["decisions"] = df["approvals"] + df["denials"]
df["approval_rate"] = np.where(df["decisions"] > 0, df["approvals"] / df["decisions"], np.nan)

employer_col = "employer_(petitioner)_name"
df["employer_clean"] = (
    df[employer_col].fillna("").astype(str).str.upper()
      .str.replace(r"[^A-Z0-9 ]", "", regex=True)
      .str.replace(r"\b(INC|LLC|LTD|CORP|CORPORATION|COMPANY|CO)\b", "", regex=True)
      .str.replace(r"\s+", " ", regex=True).str.strip()
)

# Exclude rows without employer name from employer ranking, but retain them in cleaned detail data
valid = df[df["employer_clean"].ne("")].copy()

employer_summary = (valid.groupby("employer_clean", as_index=False)
    .agg(approvals=("approvals", "sum"),
         denials=("denials", "sum"),
         decisions=("decisions", "sum"),
         states=("petitioner_state", "nunique"),
         cities=("petitioner_city", "nunique")))
employer_summary["approval_rate"] = np.where(
    employer_summary["decisions"] > 0,
    employer_summary["approvals"] / employer_summary["decisions"], np.nan
)
employer_summary = employer_summary.sort_values("approvals", ascending=False)

state_summary = (valid.groupby("petitioner_state", dropna=False, as_index=False)
    .agg(approvals=("approvals", "sum"),
         denials=("denials", "sum"),
         decisions=("decisions", "sum"),
         employers=("employer_clean", "nunique")))
state_summary["approval_rate"] = np.where(
    state_summary["decisions"] > 0,
    state_summary["approvals"] / state_summary["decisions"], np.nan
)
state_summary = state_summary.sort_values("approvals", ascending=False)

city_summary = (valid.groupby(["petitioner_state", "petitioner_city"], dropna=False, as_index=False)
    .agg(approvals=("approvals", "sum"),
         denials=("denials", "sum"),
         decisions=("decisions", "sum"),
         employers=("employer_clean", "nunique")))
city_summary["approval_rate"] = np.where(
    city_summary["decisions"] > 0,
    city_summary["approvals"] / city_summary["decisions"], np.nan
)
city_summary = city_summary.sort_values("approvals", ascending=False)

industry_summary = (valid.groupby("industry_(naics)_code", dropna=False, as_index=False)
    .agg(approvals=("approvals", "sum"),
         denials=("denials", "sum"),
         decisions=("decisions", "sum"),
         employers=("employer_clean", "nunique")))
industry_summary["approval_rate"] = np.where(
    industry_summary["decisions"] > 0,
    industry_summary["approvals"] / industry_summary["decisions"], np.nan
)
industry_summary = industry_summary.sort_values("approvals", ascending=False)

files = {
    "h1b_cleaned.csv": df,
    "employer_summary.csv": employer_summary,
    "state_summary.csv": state_summary,
    "city_summary.csv": city_summary,
    "industry_summary.csv": industry_summary,
}
for name, table in files.items():
    table.to_csv(os.path.join(outdir, name), index=False, encoding="utf-8-sig")

zip_path = "/mnt/data/h1b_powerbi_files.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in files:
        z.write(os.path.join(outdir, name), arcname=name)

print("Rows:", len(df))
print("Fiscal years:", sorted(df["fiscal_year"].dropna().unique().tolist()))
print("Total approvals:", int(df["approvals"].sum()))
print("Total denials:", int(df["denials"].sum()))
print("Created:", zip_path)

Rows: 65472
Fiscal years: [2026]
Total approvals: 395176
Total denials: 11990
Created: /mnt/data/h1b_powerbi_files.zip
